In [ ]:
!pip install wikipedia

In [ ]:
!pip install beautifulsoup4

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
import wikipedia
import requests
from bs4 import BeautifulSoup
import re

In [ ]:
df=pd.read_csv('Video_Games_Sales_Raw.csv')
df.sample(5)

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
11350,11352,Metal Slug Advance,GBA,2004.0,Shooter,Ignition Entertainment,0.06,0.02,0.0,0.00,0.08
2609,2611,WCW Mayhem,N64,1999.0,Fighting,Electronic Arts,0.63,0.15,0.0,0.01,0.79
16383,16386,Pilot Academy,PSP,2006.0,Simulation,Rising Star Games,0.00,0.01,0.0,0.00,0.01
12208,12210,Cold Stone Creamery: Scoop It Up,Wii,2009.0,Misc,Zoo Games,0.06,0.00,0.0,0.00,0.07
8204,8206,NBA Starting Five,PS2,NaN,Sports,Unknown,0.09,0.07,0.0,0.02,0.18


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16598 entries, 0 to 16597
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rank          16598 non-null  int64  
 1   Name          16598 non-null  object 
 2   Platform      16598 non-null  object 
 3   Year          16327 non-null  float64
 4   Genre         16598 non-null  object 
 5   Publisher     16540 non-null  object 
 6   NA_Sales      16598 non-null  float64
 7   EU_Sales      16598 non-null  float64
 8   JP_Sales      16598 non-null  float64
 9   Other_Sales   16598 non-null  float64
 10  Global_Sales  16598 non-null  float64
dtypes: float64(6), int64(1), object(4)
memory usage: 1.4+ MB


In [ ]:
df[df['Year'].isnull()]

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
179,180,Madden NFL 2004,PS2,NaN,Sports,Electronic Arts,4.26,0.26,0.01,0.71,5.23
377,378,FIFA Soccer 2004,PS2,NaN,Sports,Electronic Arts,0.59,2.36,0.04,0.51,3.49
431,432,LEGO Batman: The Videogame,Wii,NaN,Action,Warner Bros. Interactive Entertainment,1.86,1.02,0.00,0.29,3.17
470,471,wwe Smackdown vs. Raw 2006,PS2,NaN,Fighting,NaN,1.57,1.02,0.00,0.41,3.00
607,608,Space Invaders,2600,NaN,Shooter,Atari,2.36,0.14,0.00,0.03,2.53
...,...,...,...,...,...,...,...,...,...,...,...
16307,16310,Freaky Flyers,GC,NaN,Racing,Unknown,0.01,0.00,0.00,0.00,0.01
16327,16330,Inversion,PC,NaN,Shooter,Namco Bandai Games,0.01,0.00,0.00,0.00,0.01
16366,16369,Hakuouki: Shinsengumi Kitan,PS3,NaN,Adventure,Unknown,0.01,0.00,0.00,0.00,0.01
16427,16430,Virtua Quest,GC,NaN,Role-Playing,Unknown,0.01,0.00,0.00,0.00,0.01


In [ ]:
df.isnull().sum()

,0
Rank,0
Name,0
Platform,0
Year,271
Genre,0
Publisher,58
NA_Sales,0
EU_Sales,0
JP_Sales,0
Other_Sales,0


#Filling Games with missing years with authentic data by using Wikipedia API

In [ ]:
def getReleaseYear(game_name,search_for):
    headers = {
        "User-Agent": "FetchVideoGameInfo(ssjmukherjee@gmail.com)"
    }
    search_url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action":"query",
        "list":"search",
        "srsearch":game_name + " video game",
        "format":"json"
    }
    #First proper video game name to be searched
    search_response = requests.get(search_url, params=params, headers=headers)
    data = search_response.json()

    #Fetching the top search result and passing that to the wiki page url. Also need to make sure to replace whitespace with _
    result = data.get("query", {}).get("search", [])
    if not result:
      print(f"No Wikipedia search results for {game_name}")
      return None
    page_title = data["query"]["search"][0]["title"]
    page_url = f"https://en.wikipedia.org/wiki/{page_title.replace(' ', '_')}"

    #Now fetching the page containing the game that we saerched
    response = requests.get(page_url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    infobox = soup.find("table", {"class": "infobox"})
    if not infobox:
      print(f"No infobox found for '{game_name}' ({page_title})")
      return None
    for row in infobox.find_all("tr"):
        header = row.find("th")
        if header and header.text.strip().lower() in [search_for, f'{search_for}d',f'{search_for}s']:  #publishers, released, releases
            value = row.find("td").get_text(separator=" ", strip=True)
            return value

    return f"No release info found for '{page_title}'."

#Looking games with null Year and populating the obtained year from Wiki
for idx, row in df[df["Year"].isna()].iterrows():
    game_name = row["Name"]
    year_description = getReleaseYear(game_name,"release")
    if year_description is None:
      continue
    year_arr=year_description.split(":")  #Most of the data has the format 'Country1 and Platform1 Relaesed Year1: Country2 and Platform2 Relaesed Year2'. Hence splittingwith
    year_pattern = re.compile(r'\b(19|20)\d{2}\b')
    year_parts = [year_str_element.strip() for year_str_element in year_arr if year_pattern.search(year_str_element)]
    if len(year_parts)>0:
      year_parts=year_parts[0]
      year = re.search(r'\b(19|20)\d{2}\b', year_parts)
      if year:
        print(year.group(0), ' ', float(year.group(0)))
        df.at[idx, "Year"] = float(year.group(0))

for idx, row in df[df["Publisher"].isna()].iterrows():
    game_name = row["Name"]
    publisher_str = getReleaseYear(game_name,"publisher")
    if publisher_str is None:
      continue
    df.at[idx, "Publisher"] = publisher_str


2003   2003.0
2008   2008.0
2004   2004.0
1978   1978.0
2007   2007.0
2001   2001.0
2008   2008.0
2003   2003.0
2007   2007.0
2025   2025.0
2007   2007.0
No infobox found for 'Triple Play 99' (Triple Play (video game series))
2011   2011.0
2008   2008.0
No infobox found for 'Adventure' (Adventure game)
1977   1977.0
2003   2003.0
2002   2002.0
2007   2007.0
1999   1999.0
1997   1997.0
2011   2011.0
1977   1977.0
2002   2002.0
2005   2005.0
2011   2011.0
2008   2008.0
2011   2011.0
2011   2011.0
2006   2006.0
2002   2002.0
2008   2008.0
2001   2001.0
2010   2010.0
2004   2004.0
2011   2011.0
2011   2011.0
2011   2011.0
2004   2004.0
2008   2008.0
2011   2011.0
2007   2007.0
2005   2005.0
2002   2002.0
2002   2002.0
2004   2004.0
2003   2003.0
1980   1980.0
2009   2009.0
1987   1987.0
2006   2006.0
1977   1977.0
2004   2004.0
1980   1980.0
2008   2008.0
1976   1976.0
2008   2008.0
2010   2010.0
2011   2011.0
2011   2011.0
2008   2008.0
2001   2001.0
2008   2008.0
1978   1978.0
2007   200

In [ ]:
df.isnull().sum()

,0
Rank,0
Name,0
Platform,0
Year,51
Genre,0
Publisher,11
NA_Sales,0
EU_Sales,0
JP_Sales,0
Other_Sales,0


Now we can remove the 49 missing Year data and 10 missing Publisher data

In [ ]:
df.dropna(subset=['Year'], inplace=True)
df.dropna(subset=['Publisher'], inplace=True)
df.isnull().sum()

,0
Rank,0
Name,0
Platform,0
Year,0
Genre,0
Publisher,0
NA_Sales,0
EU_Sales,0
JP_Sales,0
Other_Sales,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16543 entries, 0 to 16597
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rank          16543 non-null  int64  
 1   Name          16543 non-null  object 
 2   Platform      16543 non-null  object 
 3   Year          16543 non-null  float64
 4   Genre         16543 non-null  object 
 5   Publisher     16543 non-null  object 
 6   NA_Sales      16543 non-null  float64
 7   EU_Sales      16543 non-null  float64
 8   JP_Sales      16543 non-null  float64
 9   Other_Sales   16543 non-null  float64
 10  Global_Sales  16543 non-null  float64
dtypes: float64(6), int64(1), object(4)
memory usage: 1.5+ MB


In [ ]:
def getReleaseYear(game_name):
    headers = {
        "User-Agent": "FetchVideoGameInfo(ssjmukherjee@gmail.com)"
    }
    search_url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action":"query",
        "list":"search",
        "srsearch":game_name + " video game",
        "format":"json"
    }
    #First proper video game name to be searched
    search_response = requests.get(search_url, params=params, headers=headers)
    data = search_response.json()

    #Fetching the top search result and passing that to the wiki page url. Also need to make sure to replace whitespace with _
    result = data.get("query", {}).get("search", [])
    if not result:
      print(f"No Wikipedia search results for {game_name}")
      return None
    page_title = data["query"]["search"][0]["title"]
    page_url = f"https://en.wikipedia.org/wiki/{page_title.replace(' ', '_')}"

    #Now fetching the page containing the game that we saerched
    response = requests.get(page_url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    infobox = soup.find("table", {"class": "infobox"})
    if not infobox:
      print(f"No infobox found for '{game_name}' ({page_title})")
      return None
    for row in infobox.find_all("tr"):
        print(row)
        header = row.find("th")
        if header and header.text.strip().lower() in ["publisher","publishers"]:
            value = row.find("td").get_text(separator=" ", strip=True)
            return value

    return f"No release info found for '{page_title}'."

#Looking games with null Year and populating the obtained year from Wiki
getReleaseYear("Witcher 3")


<tr><th class="infobox-above fn" colspan="2" style="font-size:125%;font-style:italic;">The Witcher</th></tr>
<tr><td class="infobox-image" colspan="2"><span class="mw-default-size" typeof="mw:File/Frameless"><a class="mw-file-description" href="/wiki/File:The_Witcher_video_game_series_logo.png"><img class="mw-file-element" data-file-height="206" data-file-width="483" decoding="async" height="124" src="//upload.wikimedia.org/wikipedia/en/thumb/5/53/The_Witcher_video_game_series_logo.png/330px-The_Witcher_video_game_series_logo.png" srcset="//upload.wikimedia.org/wikipedia/en/thumb/5/53/The_Witcher_video_game_series_logo.png/435px-The_Witcher_video_game_series_logo.png 1.5x, //upload.wikimedia.org/wikipedia/en/5/53/The_Witcher_video_game_series_logo.png 2x" width="290"/></a></span><div class="infobox-caption">Logo since 2015</div></td></tr>
<tr><th class="infobox-label" scope="row" style="white-space:nowrap;padding-right:0.65em;"><a href="/wiki/Video_game_genre" title="Video game genre">

'CD Projekt Atari (2007–2011) Warner Bros. Games (2012-present) Bandai Namco Entertainment (2012-present) Hands-On Mobile Spokko'

In [ ]:
df.dropna(subset=['Name'], inplace=True)
df.isnull().sum()

,0
Rank,0
Name,0
Platform,0
Year,0
Genre,0
Publisher,0
NA_Sales,0
EU_Sales,0
JP_Sales,0
Other_Sales,0


Converting Year into Integer

In [ ]:
df['Year']=df['Year'].astype(int)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16543 entries, 0 to 16597
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rank          16543 non-null  int64  
 1   Name          16543 non-null  object 
 2   Platform      16543 non-null  object 
 3   Year          16543 non-null  int64  
 4   Genre         16543 non-null  object 
 5   Publisher     16543 non-null  object 
 6   NA_Sales      16543 non-null  float64
 7   EU_Sales      16543 non-null  float64
 8   JP_Sales      16543 non-null  float64
 9   Other_Sales   16543 non-null  float64
 10  Global_Sales  16543 non-null  float64
dtypes: float64(5), int64(2), object(4)
memory usage: 1.5+ MB


In [ ]:
df['Platform'].unique()

array(['Wii', 'NES', 'GB', 'DS', 'X360', 'PS3', 'PS2', 'SNES', 'GBA',
       '3DS', 'PS4', 'N64', 'PS', 'XB', 'PC', '2600', 'PSP', 'XOne', 'GC',
       'WiiU', 'GEN', 'DC', 'PSV', 'SAT', 'SCD', 'WS', 'NG', 'TG16',
       '3DO', 'GG', 'PCFX'], dtype=object)

We have to group the platforms

In [ ]:
platformCategory={
    "Nintendo":["NES","SNES","N64","Wii","WiiU","GB","GBA","GC","DS","3DS"],
    "PS":["PS","PS3","PSP","PS4","PS2","PSV"],
    "XBox":["X360","XB","XOne"],
    "PC":["PC", "2600", "3DO", "NG", "TG16", "WS", "PCFX"],
    "Sega":["GEN", "SCD", "SAT", "DC","GG"]
    }
#We need to re-order the platformCategory into Platform:Category
platformCategory={platform:platGroup for platGroup,platformList in platformCategory.items() for platform in platformList}
platformCategory

{'NES': 'Nintendo',
 'SNES': 'Nintendo',
 'N64': 'Nintendo',
 'Wii': 'Nintendo',
 'WiiU': 'Nintendo',
 'GB': 'Nintendo',
 'GBA': 'Nintendo',
 'GC': 'Nintendo',
 'DS': 'Nintendo',
 '3DS': 'Nintendo',
 'PS': 'PS',
 'PS3': 'PS',
 'PSP': 'PS',
 'PS4': 'PS',
 'PS2': 'PS',
 'PSV': 'PS',
 'X360': 'XBox',
 'XB': 'XBox',
 'XOne': 'XBox',
 'PC': 'PC',
 '2600': 'PC',
 '3DO': 'PC',
 'NG': 'PC',
 'TG16': 'PC',
 'WS': 'PC',
 'PCFX': 'PC',
 'GEN': 'Sega',
 'SCD': 'Sega',
 'SAT': 'Sega',
 'DC': 'Sega',
 'GG': 'Sega'}

In [ ]:
df['Platform Group']=df['Platform'].map(platformCategory)
df.head()

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Platform Group
0,1,Wii Sports,Wii,2006,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74,Nintendo
1,2,Super Mario Bros.,NES,1985,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,Nintendo
2,3,Mario Kart Wii,Wii,2008,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82,Nintendo
3,4,Wii Sports Resort,Wii,2009,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00,Nintendo
4,5,Pokemon Red/Pokemon Blue,GB,1996,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,Nintendo


In [ ]:
df.isnull().sum()

,0
Rank,0
Name,0
Platform,0
Year,0
Genre,0
Publisher,0
NA_Sales,0
EU_Sales,0
JP_Sales,0
Other_Sales,0


In [ ]:
col_position = df.columns.get_loc('Platform') + 1  # right after 'Platform'
df.insert(col_position, 'Platform Group', df.pop('Platform Group'))

In [ ]:
df.columns

Index(['Rank', 'Name', 'Platform', 'Platform Group', 'Year', 'Genre',
       'Publisher', 'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales',
       'Global_Sales'],
      dtype='object')

Renaming the columns

In [ ]:
df.rename(columns={'NA_Sales':'NA Sales','EU_Sales':'EU Sales','JP_Sales':'JP Sales','Other_Sales':'Other Sales','Global_Sales':'Global Sales','Critic_Score':'Critic Score','Critic_Count':'Critic Count','User_Score':'User Score','User_Count':'User Count'}, inplace=True)
df.columns

Index(['Rank', 'Name', 'Platform', 'Platform Group', 'Year', 'Genre',
       'Publisher', 'NA Sales', 'EU Sales', 'JP Sales', 'Other Sales',
       'Global Sales'],
      dtype='object')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16543 entries, 0 to 16597
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Rank            16543 non-null  int64  
 1   Name            16543 non-null  object 
 2   Platform        16543 non-null  object 
 3   Platform Group  16543 non-null  object 
 4   Year            16543 non-null  int64  
 5   Genre           16543 non-null  object 
 6   Publisher       16543 non-null  object 
 7   NA Sales        16543 non-null  float64
 8   EU Sales        16543 non-null  float64
 9   JP Sales        16543 non-null  float64
 10  Other Sales     16543 non-null  float64
 11  Global Sales    16543 non-null  float64
dtypes: float64(5), int64(2), object(5)
memory usage: 1.6+ MB


Converting Year of Release into Integer

In [ ]:
df['Year']=df['Year'].astype(int)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16543 entries, 0 to 16597
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Rank            16543 non-null  int64  
 1   Name            16543 non-null  object 
 2   Platform        16543 non-null  object 
 3   Platform Group  16543 non-null  object 
 4   Year            16543 non-null  int64  
 5   Genre           16543 non-null  object 
 6   Publisher       16543 non-null  object 
 7   NA Sales        16543 non-null  float64
 8   EU Sales        16543 non-null  float64
 9   JP Sales        16543 non-null  float64
 10  Other Sales     16543 non-null  float64
 11  Global Sales    16543 non-null  float64
dtypes: float64(5), int64(2), object(5)
memory usage: 1.6+ MB


In [ ]:
df.to_csv("Video_Games_Sales_Cleaned_1.csv", index=False)
files.download("Video_Games_Sales_Cleaned_1.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>